# 05｜单件、多个与当前视野计数

目标：在真实分散零件照片中提取连通域、按当前画面编号，并理解“连通域数量”和“物理零件数量”的差别。原作者说明六颗螺钉、两颗小金属球，因此本例全部可见零件为 8 个；如果任务只数螺钉，真值应为 6，不能偷换口径。

编号只表示当前帧的顺序，不是跨帧追踪身份，也不是预先填好的 2×4 固定位置。

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_utils.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import lab_utils as lab
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

In [ ]:
image = lab.load_case("parts_separated")
g = lab.gray(image)
# 本照片背景集中在中灰；同时取暗轮廓与亮反光，参数仅针对这个受控案例。
raw = np.where((g < 110) | (g > 190), 255, 0).astype(np.uint8)
# 原照片最右有一条白边，排除贴图像边界的成分，不能当成零件。
raw[:, -2:] = 0
kernel = np.ones((13, 3), np.uint8)  # 竖向连断纹，过宽会合并邻件
closed = cv2.morphologyEx(raw, cv2.MORPH_CLOSE, kernel)
boxes = lab.component_boxes(closed, min_area=200)
boxes = [b for b in boxes if b["bbox"][2] > 5 and b["bbox"][3] > 5]
lab.show_images([image, closed, lab.draw_boxes(image, boxes)],
                ["真实照片：6 螺钉 + 2 球", "派生候选二值图", f"算法观察到 {len(boxes)} 个区域，黄色待复核"], cols=3)
print("人工口径：全部可见零件 8；算法区域：", len(boxes), "；差值：", len(boxes) - 8)

## 1. 看每个区域，不只看总数

总数相同仍可能同时漏一件、多一件。先显示每件框、面积和中心，再人工检查一一对应。当前函数按画面行和横坐标排序；若要稳定追踪同一个零件，需另加关联逻辑，不能把当前编号存成工件 ID。

In [ ]:
for index, row in enumerate(boxes, 1):
    print(f"#{index}", "bbox(px)=", row["bbox"], "area=", row["area"])
if boxes:
    x, y, width, height = boxes[0]["bbox"]
    crop = image[y:y + height, x:x + width]
    lab.show_images([crop], ["由当前检测框切出的单件（需人工复核框是否完整）"], cols=1)

## 2. 先把变化背景扣掉，再观察受控小视野

上面的全局灰度范围产生粘连。现在用大窗口中值滤波估计缓变背景，再看偏离背景的亮暗像素。这里额外展示左侧人工选取的四个可见零件，让你比较局部成功与全景失败；这个人工裁切只是教学，实际系统必须由汇流排定位器产生 ROI，不能写死固定位置冒充实时检测。

In [ ]:
background = cv2.medianBlur(g, 51)
contrast = cv2.absdiff(g, background)
local_mask = np.where(contrast > 25, 255, 0).astype(np.uint8)
local_mask[:, -3:] = 0
local_mask = cv2.morphologyEx(local_mask, cv2.MORPH_CLOSE, np.ones((7, 3), np.uint8))
full_candidates = lab.component_boxes(local_mask, min_area=150)
left_roi = image[:, :300]  # 教学人工 ROI：两螺钉 + 两金属球，非生产定位逻辑
left_candidates = lab.component_boxes(local_mask[:, :300].copy(), min_area=150)
lab.show_images([background, lab.draw_boxes(image, full_candidates), lab.draw_boxes(left_roi, left_candidates)],
    ["派生：估计缓变背景", f"全图候选 {len(full_candidates)} / 真值 8：仍需复核",
     f"人工教学小视野 {len(left_candidates)} / 真值 4"], cols=3)
print("受控 ROI 的区域数量：", len(left_candidates), "；整图仍有两件粘连，不能绿灯。")
assert len(left_candidates) == 4  # 只验证这张公开教学照片的可重复性，不代表独立准确率

## 3. 核大小会改变数量

本例的亮暗反光使一个螺钉可能碎成多个区域；过宽的闭运算又可能把两颗相邻螺钉合在一起。不要自动挑一个“刚好等于 8”的参数后把它作为可靠性证明。

In [ ]:
views, labels = [], []
for kw, kh in [(3, 3), (3, 13), (15, 15)]:
    candidate = cv2.morphologyEx(raw, cv2.MORPH_CLOSE, np.ones((kh, kw), np.uint8))
    candidates = lab.component_boxes(candidate, min_area=200)
    views.append(lab.draw_boxes(image, candidates))
    labels.append(f"核 {kw}×{kh}：{len(candidates)} 个区域")
lab.show_images(views, labels, cols=3)
display(HTML(lab.parameter_gallery(views, labels, "count")))

## 迁移到刀片电池

先检测当前视野中实际可见汇流排，再分割每块上的镍片。N×M 只可作配方中的预期数量，不能代替当前图像的实测数量。被遮挡、出画或粘连的目标应标出原因并待复核。多个摄像头若视野重叠，先校准和去重，再算整包数量。

## 来源与练习边界

公开焊缝：sukmaadhiwijaya / rikkarth，固定提交 `8b80d1afe88c0b946500a498cbdb9d7a1eda5533`，数据卡声明 CC0-1.0；工业表面：Kolektor Group / ViCoS，CC BY-NC-SA 4.0；分散零件：Tung1213，CC BY-SA 3.0；重叠零件：Technologyfelix，CC BY-SA 4.0。相关照片和派生图保留原许可及署名。

完整来源、原许可页、下载路径及 SHA256 见 [DATA_SOURCES.md](../DATA_SOURCES.md) 和 [source_manifest.json](../sources/source_manifest.json)。本课是非商业教学。灰度、二值、叠框及扰动均为原照片的派生图，七格合成图另有醒目标记。它们不等于新增独立验收数据，也不证明电池焊接准确率或机械强度。